In [6]:
%pip install python-dotenv requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***")

6B95***


In [4]:
import requests
def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    url = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word"
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()  
    return r.json()

Q(1) a에 대한 설명:
requests 라이브러리를 사용해 우리말샘 API에 GET 요청을 보내고 결과를 JSON 형태로 받습니다. timeout=10을 설정해서 무한정 기다리지 않게 했고, raise_for_status()를 통해 HTTP 오류를 검사했습니다.

In [8]:
import json
data = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


Q1(b) 에 대한 설명:
ensure_ascii=False를 빼면 온전한 한글 테스트가 아닌 유니코드 이스케이프 문자 형태로 인코딩되어 출력됩니다. 

In [9]:
channel = data.get("channel", {})
total = channel.get("total", 0)
items = channel.get("item", [])

print(f"총 {total}건, 이 페이지 {len(items)}건")

for item in items[:5]:
    word = item.get("word", "")
    pos = item.get("pos") or "품사 없음" 
    
    sense = item.get("sense", {})
    if isinstance(sense, list) and len(sense) > 0:
        definition = sense[0].get("definition", "")
    elif isinstance(sense, dict):
        definition = sense.get("definition", "")
    else:
        definition = ""
        
    print(f"{word} ({pos}) -> {definition[:40]}")

총 328건, 이 페이지 10건
김치 (품사 없음) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) -> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


Q1(c) 에 대한 설명:
PI 응답 딕셔너리에서 전체 검색 결과 수와 현재 페이지의 항목 수를 찾아 지정된 형식으로 출력했습니다. 첫 5개 항목을 추출할 때 dict.get() 메서드를 활용하여, 품사(pos) 정보가 누락된 항목이 있더라도 코드 에러 없이 "품사 없음"으로 안전하게 처리되도록 작성했습니다.

In [11]:
import time
from collections import Counter

words: list[str] = [
    "김치", "라면", "만두", "김밥", 
    "국수", "떡볶이", "불고기", "비빔밥"
]

all_items = []

# (a) 검색어별 결과 수 출력
for q in words:
    res = search_word(q)
    total = res.get("channel", {}).get("total", 0)
    print(f"{q}: {total}건")
    items = res.get("channel", {}).get("item", [])
    all_items.extend(items)
    
    time.sleep(0.3)  

print("\n" + "="*30 + "\n")

# (b) 품사 빈도 상위 3개 추출
pos_list = [item.get("pos") or "(미상)" for item in all_items]
pos_counts = Counter(pos_list)

print(pos_counts.most_common(3))

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


[('(미상)', 80)]


Q2(a)에 대한 설명:
검색어를 반복문으로 돌면서 앞서 만든 search_word 함수를 호출해 각 단어의 전체 검색 결과 수를 출력했습니다. 서버에 부담을 주지 않도록 time.sleep(0.3)을 넣어 0.3초씩 대기 시간을 두었습니다.

Q2(b)에 대한 설명:
모든 검색어의 응답 항목을 하나의 리스트에 합친 후, collections.Counter를 사용해 가장 많이 등장한 품사 3개의 빈도를 빠르고 간편하게 추출했습니다.

관찰: 가장 흔한 품사는 미상으로 집계되었지만 '명사'일 것입니다. 제시된 8개의 검색어 모두 명사이기 때문입니다.